# 22. Recursive Retrosynthetic Solver

Standard MCTS-based retrosynthetic planning exhausts its time budget in a
single top-down search.  For hard molecules it often gets close: finding
partial routes that only fail because one small fragment cannot be
decomposed. Currently, the search has no mechanism to recover from those dead ends.

This notebook implements a **recursive blocker-resolution** strategy to recover
from those near-misses:

1. Run the main MCTS search on the target molecule.  If it finds a complete
   route, return it immediately.
2. If not, examine the search tree for **sole blockers**: frontier leaf nodes
   that have exactly one unsolved precursor.  If that precursor could be solved,
   the entire branch through that leaf would complete.
3. Pick the smallest (molecular weight or heavy atoms number) sole blocker (easiest to solve) and **recursively** run
   a fresh MCTS search on it as a sub-target.
4. If the sub-search succeeds, **merge** the partial route (target → blocker)
   with the sub-route (blocker → building blocks) into a single connected graph.
5. Repeat up to `MAX_RECURSION_DEPTH` levels.  A `fragment_cache` avoids
   re-solving the same fragment, and an `in_progress` set detects cycles.

The current example uses a overall **time budget** of 10 minutes for fair comparision. 
The worst-case total time is `MAIN_TIME + MAX_RECURSION_DEPTH × SUB_TIME`
(e.g. 2 min + 4 × 2 min = 10 min with the default settings).

In [1]:
import gc           # garbage collection used to free sub-search trees
import json
import time
from itertools import pairwise  # requires Python 3.10+
from pathlib import Path

## 2. Download Data and Load Resources

SynPlanner ships with a preset (`synplanner-article`) that contains:
- **Reaction rules** (24k SMARTS-based retrosynthetic transformations).
- **Building blocks** (~186k commercially available fragments; molecules
  with ≤ `min_mol_size` heavy atoms are also treated as building blocks).
- **Filtering policy** weights: a GCN that predicts rule applicability.
- **Ranking policy** weights: a GCN that ranks applicable rules by likelihood.

All four are downloaded once to `synplan_data/` and cached for subsequent runs.

In [2]:
from synplan.utils.config import RDKitEvaluationConfig
from synplan.utils.loading import (
    download_preset,
    load_building_blocks,
    load_evaluation_function,
    load_reaction_rules,
)

# Download all preset data to synplan_data/ (no-op if already cached).
# Returns a dict of paths: "building_blocks", "reaction_rules",
# "filtering_policy", "ranking_policy".
paths = download_preset("synplanner-article", save_to=Path("synplan_data"))

building_blocks = load_building_blocks(
    paths["building_blocks"], standardize=True, silent=True
)
reaction_rules = load_reaction_rules(paths["reaction_rules"])
# evaluation_function: scores a node by the maximum heavy-atom count of its
# unsolved precursors, which guides UCT to prefer simpler/smaller fragments.
evaluation_function = load_evaluation_function(
    RDKitEvaluationConfig(score_function="heavyAtomCount") 
)

print(f"Building blocks : {len(building_blocks)}")
print(f"Reaction rules  : {len(reaction_rules)}")

Building blocks : 186030
Reaction rules  : 24094


## 3. Configure the Combined Policy Network

The combined policy merges two neural networks:

- **Filtering policy**: a multi-label classifier that predicts which reaction
  rules are *applicable* to the current molecule.  Rules below
  `filtering_threshold` are vetoed before the ranking step.
- **Ranking policy**: predicts which of the surviving rules is most likely to
  lead to a complete route.  The softmax probabilities are used as the prior
  for the UCT exploration score.

**Key parameters:**
- `top_rules`: keep only the top-N rules after ranking (caps search branching
  factor).
- `filtering_threshold`: hard veto on the filtering score. Lower values allow
  more rules through, higher values are more selective.
- `temperature`: softmax temperature for the ranking distribution. Higher
  values flatten it toward uniform, lower values sharpen it.

In [3]:
# ── Hyperparameters (edit these directly) ──────────────────────────────────
# These are also available in per-bin Bayesian-optimised YAML configs
# (see configs/bayesian/); run the optional cell below to load those instead.
mcts_params = {
    "c_ucb": 0.1,              # UCT exploration constant — higher = more exploration
    "max_iterations": 100000,  # hard cap on MCTS iterations (time limit usually triggers first)
    "max_depth": 15,           # maximum retrosynthetic depth
    "evaluation_agg": "max",   # how to aggregate child scores when evaluating a node
}

policy_params = {
    "top_rules": 50,             # keep only the top-N rules after ranking (caps branching factor)
    "filtering_threshold": 0.01, # filtering score veto threshold (0.0 = no veto)
    "temperature": 1.0,          # softmax temperature for the ranking distribution
}

print("Hyperparameters set:")
print(f"  c_ucb              : {mcts_params['c_ucb']}")
print(f"  top_rules          : {policy_params['top_rules']}")
print(f"  filtering_threshold: {policy_params['filtering_threshold']:.2e}")

Hyperparameters set:
  c_ucb              : 0.1
  top_rules          : 50
  filtering_threshold: 1.00e-02


### Optional: load per-bin Bayesian-optimised hyperparameters

If you are running the SA-score benchmark (see `scripts/benchmark/`), each
SA-score bin has its own Bayesian-optimised config in `configs/bayesian/`.
The four variant files (`_a` through `_d`) for a given bin share the same UCT
hyperparameters and only differ in `phase1.max_time`, which this notebook
overrides with `MAIN_TIME` anyway.  Any variant file works; `_a` is used here.

In [ ]:
# ── (Optional) Load from per-bin Bayesian-optimised config ─────────────────
# Uncomment and run this cell to override the manual hyperparameters above,
# then re-run the cells that build policy_function, main_config, and sub_config.
# All four variant files (_a through _d) share the same UCT hyperparameters
# for a given bin — pick any one.

# import yaml

# BIN_LABEL = "3.5_4.5"
#
# config_path = Path(f"configs/bayesian/config_{BIN_LABEL}_a.yaml")
# with open(config_path) as f:
#     raw = yaml.safe_load(f)
#
# phase1 = raw["phase1"]
# mcts_params["c_ucb"]                 = phase1["c_ucb"]
# mcts_params["max_iterations"]        = phase1.get("max_iterations", 100_000)
# mcts_params["max_depth"]             = phase1.get("max_depth", 15)
# policy_params["top_rules"]           = phase1["top_rules"]
# policy_params["filtering_threshold"] = phase1["filtering_threshold"]
# policy_params["temperature"]         = phase1.get("temperature", 1.0)
#
# # Note: phase1.max_time is ignored — MAIN_TIME controls the search budget.
# print(f"Loaded config for bin {BIN_LABEL}:")
# print(f"  c_ucb              : {mcts_params['c_ucb']:.6f}")
# print(f"  top_rules          : {policy_params['top_rules']}")
# print(f"  filtering_threshold: {policy_params['filtering_threshold']:.2e}")


In [4]:
from synplan.mcts.expansion import CombinedPolicyNetworkFunction
from synplan.utils.config import PolicyNetworkConfig

# CombinedPolicyNetworkFunction wraps both networks and applies them in sequence:
#   1. Filtering network scores all rules; rules below filtering_threshold are dropped.
#   2. Ranking network scores the survivors and returns a probability distribution.
#   3. top_rules: only the top-N rules (by ranking score) are returned as expansion candidates.
policy_function = CombinedPolicyNetworkFunction(
    filtering_config=PolicyNetworkConfig(
        weights_path=str(paths["filtering_policy"]),
        policy_type="filtering",
    ),
    ranking_config=PolicyNetworkConfig(
        weights_path=str(paths["ranking_policy"]),
        policy_type="ranking",
    ),
    top_rules=policy_params["top_rules"],
    rule_prob_threshold=0.0,   # additional probability floor (0 = disabled)
    ranking_weight=1.0,        # weight of the ranking score relative to the filtering score
    temperature=policy_params.get("temperature", 1.0),
    filtering_threshold=policy_params["filtering_threshold"],
)

print("Combined policy loaded.")

Lightning automatically upgraded your loaded checkpoint from v1.9.5 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint synplan_data/policy/supervised_gcn/v1/v1/filtering_policy.ckpt`
Lightning automatically upgraded your loaded checkpoint from v1.9.5 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint synplan_data/policy/supervised_gcn/v1/v1/ranking_policy.ckpt`


Combined policy loaded.


## 4. Configure the Tree Search

We create two `TreeConfig` objects: 
one for the initial search on the target
molecule, and one for each recursive sub-search on a blocker fragment.

- **`main_config`** (`MAIN_TIME` seconds): wider exploration budget, run once.
- **`sub_config`** (`SUB_TIME` seconds): same hyperparameters, used at each
  recursion level.

**Worst-case time**: `MAIN_TIME + MAX_RECURSION_DEPTH × SUB_TIME`
(default: 2 + 4 × 2 = **10 min** per molecule).
Reduce `MAX_RECURSION_DEPTH` or `SUB_TIME` for faster (less thorough) runs.

In [5]:
from synplan.utils.config import TreeConfig

# Time budgets and recursion.
MAIN_TIME = 120          # seconds for the initial search on the target molecule
SUB_TIME = 120           # seconds for each recursive sub-search on a blocker
MAX_RECURSION_DEPTH = 4  # maximum number of recursive levels (0 = no recursion)
MAX_SOLE_BLOCKERS = 1    # how many sole-blocker candidates to try per recursion level

# main_config: used for the initial search.  Uses the UCT algorithm with
# 'evaluation_first' search strategy (expand the highest-scored precursor first).
main_config = TreeConfig(
    search_strategy="evaluation_first",
    max_iterations=mcts_params.get("max_iterations", 100000),
    max_time=MAIN_TIME,
    max_depth=mcts_params.get("max_depth", 15),
    min_mol_size=6,         # molecules with <= 6 heavy atoms are treated as building blocks
    init_node_value=0.5,    # initial value for unexplored nodes
    ucb_type="uct",
    c_ucb=mcts_params["c_ucb"],
    evaluation_agg=mcts_params.get("evaluation_agg", "max"),
    silent=True,
)

# sub_config: identical to main_config except for the time limit.
# Using the same hyperparameters keeps sub-searches comparable to the main search.
sub_config = TreeConfig(
    search_strategy="evaluation_first",
    max_iterations=mcts_params.get("max_iterations", 100000),
    max_time=SUB_TIME,
    max_depth=mcts_params.get("max_depth", 15),
    min_mol_size=6,
    init_node_value=0.5,
    ucb_type="uct",
    c_ucb=mcts_params["c_ucb"],
    evaluation_agg=mcts_params.get("evaluation_agg", "max"),
    silent=True,
)

print(f"Main search: {MAIN_TIME}s, Sub search: {SUB_TIME}s")
print(f"Max recursion depth: {MAX_RECURSION_DEPTH}, Max sole blockers: {MAX_SOLE_BLOCKERS}")

Main search: 120s, Sub search: 120s
Max recursion depth: 4, Max sole blockers: 1


## 5. Run MCTS on a Single Molecule

We parse the target SMILES and run a standard MCTS search using the
`main_config`.

After the search:
- `tree.winning_nodes` lists the IDs of nodes that represent complete routes
  (all precursors at that node are building blocks).
- `tree.to_stats_dict()` returns iteration count, node count, etc.

If the search is not solved, we proceed to blocker resolution in the next
sections. **The tree is kept in memory** so the recursive solver can reuse it.

In [6]:
from synplan.chem.utils import mol_from_smiles
from synplan.mcts.tree import Tree

# ── Target molecule ─────────────────────────────────────────────────────────
# Paste any SMILES here.  This molecule is intentionally chosen as a hard case
# that the standard MCTS search cannot solve within the time budget, so the
# recursive solver's benefit is clearly demonstrated.
target_smiles = "CCC(C)N1C(=O)C(=CNNc2nc3c(c(=O)n(C)c(=O)n3C)n2C)C(=O)NC1=S"

# ── Or load from a benchmark file ───────────────────────────────────────────
# BIN_LABEL = "3.5_4.5"
# smi_file = Path("sascore") / f"targets_with_sascore_{BIN_LABEL}.smi"
# smiles_list = [l.strip().split()[0] for l in smi_file.read_text().splitlines() if l.strip()]
# target_smiles = smiles_list[0]

mol = mol_from_smiles(target_smiles, standardize=True, clean2d=True, clean_stereo=True)
print(f"Target: {target_smiles}")
print(f"Atoms:  {len(mol)}")
mol  # display 2D structure in Jupyter

Target: CCC(C)N1C(=O)C(=CNNc2nc3c(c(=O)n(C)c(=O)n3C)n2C)C(=O)NC1=S
Atoms:  30


In [7]:
tree = Tree(
    target=mol,
    config=main_config,
    reaction_rules=reaction_rules,
    building_blocks=building_blocks,
    expansion_function=policy_function,
    evaluation_function=evaluation_function,
)

# The Tree object is an iterator: each step runs one MCTS iteration.
# Iterating to exhaustion runs until max_time or max_iterations is reached.
t0 = time.time()
for _ in tree:
    pass
elapsed = time.time() - t0

stats = tree.to_stats_dict()
n_routes = len(tree.winning_nodes)  # winning_nodes: IDs of nodes where all precursors are in stock

print(f"Result:     {'SOLVED' if n_routes > 0 else 'NOT SOLVED'}")
print(f"Time:       {elapsed:.1f}s")
print(f"Iterations: {stats['num_iter']}")
print(f"Nodes:      {stats['num_nodes']}")
print(f"Routes:     {n_routes}")

Result:     NOT SOLVED
Time:       120.4s
Iterations: 373
Nodes:      12436
Routes:     0


## 6. Extract Route Graphs

A **route graph** is a flat dictionary `{parent_smiles: [child_smiles, ...]}`
where each entry represents one retrosynthetic step: the parent molecule
was decomposed into the child fragments.

`tree.route_to_node(node_id)` walks up the `parents` dict from the given node
to the root and returns the path as a list of `Node` objects (root first).
`pairwise` then gives us consecutive `(parent_node, child_node)` pairs, and
for each pair:
- `before.curr_precursor`: the molecule that was expanded at the parent node.
- `after.new_precursors`: the fragments produced by the retrosynthetic reaction.

The resulting dict can be used directly for visualisation or saved to disk.
Route graphs from different sub-searches are merged by dict union (`{**a, **b}`).

In [8]:
def extract_route_graph(tree, node_id):
    """Extract a flat route graph from root to the given winning node."""
    nodes_path = tree.route_to_node(node_id)
    graph = {}
    for before, after in pairwise(nodes_path):
        graph[str(before.curr_precursor)] = [str(p) for p in after.new_precursors]
    return graph


if tree.winning_nodes:
    route = extract_route_graph(tree, tree.winning_nodes[0])
    print(f"Route has {len(route)} retrosynthetic steps:")
    for parent, children in route.items():
        print(f"  {parent[:60]}..." if len(parent) > 60 else f"  {parent}")
        for child in children:
            print(f"    -> {child[:60]}..." if len(child) > 60 else f"    -> {child}")
else:
    print("No route found, we will attempt blocker resolution in the next section.")

No route found, we will attempt blocker resolution in the next section.


## 7. Identify Sole Blockers

After the main search fails, the tree's frontier contains unexpanded leaf
nodes, each holding a queue of molecules still to be decomposed
(`precursors_to_expand`).  A node with exactly **one** unsolved precursor
(i.e. one molecule that is neither a building block nor already broken down)
is a **sole-blocker node**: if we could solve that one fragment, the entire
branch through that node would complete.

We collect all such sole-blocker fragments across the tree frontier, then
sort them by heavy atom count ascending (theoretically: smaller = easier to solve) and by
occurrence count descending (more occurrences = higher leverage).

In [9]:
def find_sole_blockers(tree):
    """Find fragments that are the sole obstacle preventing route completion.

    Only unexpanded (frontier) leaf nodes are examined.  Expanded nodes have
    stale `precursors_to_expand` data (the queue before they were processed),
    so checking them would produce false sole-blocker hits and could yield an
    incorrect partial route when the extracted path ends at an already-expanded
    internal node.

    Returns a list of dicts sorted by atom count (ascending), each containing:
    - smiles: the blocker SMILES
    - n_atoms: heavy atom count
    - nodes: list of (node_id, depth) where this blocker appears
    - count: number of tree nodes blocked by this fragment
    """
    sole_blockers = {}
    for nid, node in tree.nodes.items():
        if nid in tree.expanded_nodes:  # only look at unexpanded nodes
            continue
        if node.is_solved():
            continue
        unsolved = [
            p for p in node.precursors_to_expand
            if not p.is_building_block(tree.building_blocks, tree.config.min_mol_size)
        ]
        if len(unsolved) == 1:     # only one precursor_to_expand = "sole blocker"
            smi = str(unsolved[0])
            depth = tree.nodes_depth.get(nid, 0)
            if smi not in sole_blockers:
                sole_blockers[smi] = {
                    "smiles": smi,
                    "n_atoms": len(unsolved[0]),
                    "nodes": [],
                    "count": 0,
                }
            sole_blockers[smi]["nodes"].append((nid, depth))
            sole_blockers[smi]["count"] += 1
    return sorted(sole_blockers.values(), key=lambda x: (x["n_atoms"], -x["count"]))  # sort by number of atoms, then most-blocking fragment
                                                                                      # should eventually be heavyAtomCount + proximity to target

blockers = find_sole_blockers(tree)
print(f"Found {len(blockers)} sole blockers:")
for i, b in enumerate(blockers[:10]):
    print(f"  {i+1}. {b['smiles'][:60]}  ({b['n_atoms']} atoms, blocks {b['count']} nodes)")

Found 3311 sole blockers:
  1. ClC(=O)C(C(Cl)=O)=CNNc1n(C)c2C(=O)N(C(=O)Nc2n1)C  (23 atoms, blocks 38 nodes)
  2. ClC(C(=CNNc1nc2NC(N(C(c2n1C)=O)C)=O)C(O)=O)=O  (23 atoms, blocks 18 nodes)
  3. ClC(=O)C(C(Cl)=O)=CNNc1n(c2C(NC(=O)N(c2n1)C)=O)C  (23 atoms, blocks 4 nodes)
  4. ClC(C(=CNNc1n(c2C(NC(=O)N(c2n1)C)=O)C)C(O)=O)=O  (23 atoms, blocks 2 nodes)
  5. C(NNc1n(c2C(N(C(=O)N(C)c2n1)C)=O)C)=C(C(=O)Cl)CO  (23 atoms, blocks 2 nodes)
  6. OC(=O)C(C(O)=O)=CNNc1n(c2C(NC(=O)N(c2n1)C)=O)C  (23 atoms, blocks 2 nodes)
  7. N(NC=C(C#N)C(=O)Cl)c1n(c2C(N(C(=O)N(C)c2n1)C)=O)C  (23 atoms, blocks 1 nodes)
  8. C1(NC(NC(=O)C1=CNNc2n(C)c3C(NC(Nc3n2)=O)=O)=S)=O  (24 atoms, blocks 50 nodes)
  9. O=C(C(=CNNc1nc2NC(N(C(c2n1C)=O)C)=O)C(Cl)=O)OC  (24 atoms, blocks 22 nodes)
  10. N(NC=C(C(=O)O)C(=O)Cl)c1n(c2C(N(C(=O)N(C)c2n1)C)=O)C  (24 atoms, blocks 21 nodes)


## 8. Check Rule Applicability

Before spending `SUB_TIME` seconds on a recursive sub-search, we do a quick
sanity check: does *any* reaction rule match the blocker fragment?  If no
rule is applicable, the fragment is not decomposable by SynPlanner's rule set
and we skip it immediately (marks it as unsolvable in the cache).

In [10]:
from synplan.chem.reaction import apply_reaction_rules

def has_applicable_rules(smi, reaction_rules):
    mol = mol_from_smiles(smi, standardize=True)
    if mol is None:
        return False
    for rule in reaction_rules:
        if next(apply_reaction_rule(mol, rule), None) is not None:
            return True
    return False

# quick sanity check, stop at first rule applicable

if blockers:
    test_blocker = blockers[0]["smiles"]
    applicable = has_applicable_rules(test_blocker, reaction_rules)
    print(f"Blocker: {test_blocker[:60]}")
    print(f"Has applicable rules: {applicable}")

Blocker: ClC(=O)C(C(Cl)=O)=CNNc1n(C)c2C(=O)N(C(=O)Nc2n1)C
Has applicable rules: True


## 9. The Recursive Solver

This is the core algorithm.  `recursive_solve` takes a SMILES and returns
either a route graph `{parent: [children]}` or `None` (unsolvable).

**Algorithm summary:**
1. **Cache check**: if this SMILES was already attempted, return cached result.
2. **Cycle guard**: if this SMILES is already on the current call stack, skip
   (prevents infinite recursion on circular fragment dependencies).
3. **Building block check**: if the molecule is stock-available, return `{}`
   (empty route = no steps needed).
4. **Depth limit**: if `depth >= MAX_RECURSION_DEPTH`, give up.
5. **MCTS search**: run (or reuse) a tree search.  If solved, extract and
   return the route graph immediately.
6. **Sole blocker resolution**: find sole blockers, skip those with no
   applicable rules, then recurse on up to `MAX_SOLE_BLOCKERS` candidates.
7. **Merge**: on success, combine `partial_graph` (target → blocker position)
   with `sub_graph` (blocker → building blocks) via dict union.

**Cache semantics**: `fragment_cache[smi] = None` means definitely unsolvable;
`fragment_cache[smi] = {}` means already a building block (zero steps);
`fragment_cache[smi] = {…}` means solved with this route graph.

**Memory**: trees for sub-searches are deleted after use (`del search_tree;
gc.collect()`) to avoid accumulating large objects during deep recursion.

In [11]:
from synplan.chem.precursor import Precursor

def recursive_solve(
    smiles,
    fragment_cache,
    in_progress,
    depth=0,
    existing_tree=None,
):
    """Recursively solve a molecule by resolving sole blockers.

    Args:
        smiles: Target SMILES string (chython canonical form).
        fragment_cache: Dict mapping SMILES -> route_graph (or None if unsolvable).
            An empty dict {} means the molecule is a building block (no steps needed).
        in_progress: Set of SMILES currently on the call stack (cycle detection).
        depth: Current recursion depth (0 = initial call on the target molecule).
        existing_tree: Pre-built search tree to reuse at depth 0 (avoids
            re-running the main MCTS search that was already done in Section 5).

    Returns:
        Route graph dict {parent_smi: [child_smi, ...]}, or None if unsolvable.
    """
    indent = "  " * depth  # visual indentation for progress messages

    # Cache check: 
    # Return immediately if this molecule was already attempted in this session.
    if smiles in fragment_cache:
        cached = fragment_cache[smiles]
        print(f"{indent}[d={depth}] CACHED ({'solved' if cached is not None else 'unsolvable'})")
        return cached
        
    # None: previously determined to be unsolvable, {} empty dict means building blocks
    
    # Abort if this SMILES is already being solved higher up the call stack
    # (a cycle can occur when a blocker's sub-search generates the original target).
    if smiles in in_progress:
        print(f"{indent}[d={depth}] CYCLE detected! Skipping")
        return None

    mol = mol_from_smiles(smiles, standardize=True, clean2d=True, clean_stereo=True)
    if mol is None:
        fragment_cache[smiles] = None
        return None

    # Building block check is skipped at depth 0 (existing_tree is not None) when an existing tree is provided
    # because the target was already confirmed to be non-stock before the main search.
    if existing_tree is None:
        prec = Precursor(mol)
        if prec.is_building_block(building_blocks, cfg.min_mol_size):
            print(f"{indent}[d={depth}] Already a building block")
            fragment_cache[smiles] = {}
            return {}  # return empty route graph

    if depth > 0 and depth >= MAX_RECURSION_DEPTH:
        print(f"{indent}[d={depth}] Max recursion depth reached")
        fragment_cache[smiles] = None
        return None # caches the fragment so that it is not retried 

    in_progress.add(smiles) # any descendant call with the same SMILES will hit the cycle guard

    # Global depth limit should also be added 

    # ── MCTS search ──────────────────────────────────────────────────────────
    if existing_tree is not None:
        # Depth 0: reuse the tree built in Section 5 instead of searching again.
        search_tree = existing_tree
        n_routes = len(search_tree.winning_nodes) # known to be 0 in stop-at-first context
        print(f"{indent}[d={depth}] Reusing existing tree ({n_routes} routes)")
    else:
        # Depth > 0: run a fresh sub-search on the blocker fragment.
        cfg = sub_config
        print(f"{indent}[d={depth}] Searching: {smiles[:60]} ({len(mol)} atoms)")
        search_tree = Tree(
            target=mol,
            config=cfg,
            reaction_rules=reaction_rules,
            building_blocks=building_blocks,
            expansion_function=policy_function,
            evaluation_function=evaluation_function,
        )
        t0 = time.time()
        for _ in search_tree:
            pass
        elapsed = time.time() - t0
        n_routes = len(search_tree.winning_nodes)
        print(f"{indent}  {'SOLVED' if n_routes else 'NOT SOLVED'} ({elapsed:.1f}s, {n_routes} routes)")

    # ── Direct solution ───────────────────────────────────────────────────────
    if n_routes > 0:
        # The search found a complete route, we extract the first winning path.
        graph = extract_route_graph(search_tree, search_tree.winning_nodes[0])
        if existing_tree is None:
            del search_tree
            gc.collect()   # frees the tree object, only when the depth < 0
        in_progress.discard(smiles) # removes from call stack guard so future siblings are not blocked
        fragment_cache[smiles] = graph
        return graph

    # ── Sole blocker resolution ───────────────────────────────────────────────
    sole_blockers = find_sole_blockers(search_tree)
    print(f"{indent}  {len(sole_blockers)} sole blockers found")

    for i, sb in enumerate(sole_blockers[:MAX_SOLE_BLOCKERS]):
        sb_smi = sb["smiles"]
        print(f"{indent}  [{i+1}] {sb_smi[:60]} ({sb['n_atoms']} atoms, {sb['count']} nodes)")

        # Skip blockers that no reaction rule can decompose. 
        if not has_applicable_rules(sb_smi, reaction_rules):
            print(f"{indent}    No applicable rules — skipping")
            fragment_cache[sb_smi] = None
            continue

        # Extract the partial route from the root of this tree down to the
        # shallowest node where the blocker appears, giving us the steps
        # from the target molecule to the point where the blocker must be solved.
        shallowest_nid = min(sb["nodes"], key=lambda x: x[1])[0]
        partial_graph = extract_route_graph(search_tree, shallowest_nid)

        # Recurse: attempt to solve the blocker as a new sub-target.
        sub_graph = recursive_solve(sb_smi, fragment_cache, in_progress, depth + 1) 

        if sub_graph is not None:
            # Merge: partial_graph covers target → blocker; sub_graph covers blocker → BBs.
            merged = {**partial_graph, **sub_graph}
            if existing_tree is None:
                del search_tree
                gc.collect()
            in_progress.discard(smiles)
            fragment_cache[smiles] = merged
            return merged

    # All blocker candidates exhausted without finding a solution.
    print(f"{indent}  All blockers exhausted.")
    if existing_tree is None:
        del search_tree
        gc.collect()
    in_progress.discard(smiles)
    fragment_cache[smiles] = None
    return None

## 10. Solve a Single Molecule

We apply `recursive_solve` to our target molecule.  We pass `existing_tree=tree`
to reuse the MCTS tree already built in Section 5.

If a route is found, `route_graph_to_json` converts the flat adjacency dict
into a nested tree structure that `get_route_svg_from_json` can render.

In [12]:
from IPython.display import SVG, display
from synplan.utils.visualisation import get_route_svg_from_json


def route_graph_to_json(route_graph, root_smi, building_blocks):
    """Convert the flat route_graph dict into a nested tree for visualisation.

    The flat format `{parent_smi: [child_smi, ...]}` is efficient for merging
    sub-routes but `get_route_svg_from_json` expects a nested node tree:
      {smiles, type: 'mol', in_stock, children: [{type: 'reaction', children: [...]}]}

    Args:
        route_graph: Dict {parent_smi: [child_smi, ...]}.
        root_smi: SMILES of the root (target) molecule.
        building_blocks: frozenset[str] of canonical SMILES for stock membership.

    Returns:
        Nested dict compatible with get_route_svg_from_json.
    """
    def build_node(smi):
        in_stock = smi in building_blocks  # True if this fragment is commercially available
        node = {"smiles": smi, "type": "mol", "in_stock": in_stock}
        if smi in route_graph:
            # This molecule has a retrosynthetic step, we can recurse into its children.
            children = [build_node(child_smi) for child_smi in route_graph[smi]]
            node["children"] = [{"type": "reaction", "children": children}]
        return node
    return build_node(root_smi)

# str(mol) gives the chython canonical SMILES, which matches the keys used
# internally by Precursor.__str__ and in the building_blocks frozenset.
canonical = str(mol)
fragment_cache = {}   # SMILES -> route_graph | None
in_progress = set()   # SMILES currently on the call stack

t0 = time.time()
route_graph = recursive_solve(
    canonical,
    fragment_cache,
    in_progress,
    existing_tree=tree,  # reuse the tree from Section 5; depth defaults to 0
)
wall_time = time.time() - t0

status = "solved" if route_graph is not None else "unsolved"
print(f"\nResult: {status.upper()}  ({wall_time:.1f}s, cache entries: {len(fragment_cache)})")

if route_graph:
    print(f"Route has {len(route_graph)} retrosynthetic steps.\n")
    route_json = {0: route_graph_to_json(route_graph, canonical, building_blocks)}
    display(SVG(get_route_svg_from_json(route_json, 0)))
else:
    print("Not solved after recursive blocker resolution.")

[d=0] Reusing existing tree (0 routes)
  3311 sole blockers found
  [1] ClC(=O)C(C(Cl)=O)=CNNc1n(C)c2C(=O)N(C(=O)Nc2n1)C (23 atoms, 38 nodes)
  [d=1] Searching: ClC(=O)C(C(Cl)=O)=CNNc1n(C)c2C(=O)N(C(=O)Nc2n1)C (23 atoms)
    NOT SOLVED (120.3s, 0 routes)
    2285 sole blockers found
    [1] Nc1c(C#N)n(C)c(NN)n1 (11 atoms, 105 nodes)
    [d=2] Searching: Nc1c(C#N)n(C)c(NN)n1 (11 atoms)
      NOT SOLVED (122.4s, 0 routes)
      6576 sole blockers found
      [1] Nc1n(C)cc(Cl)n1 (8 atoms, 17 nodes)
      [d=3] Searching: Nc1n(C)cc(Cl)n1 (8 atoms)
        SOLVED (120.4s, 211 routes)

Result: SOLVED  (380.2s, cache entries: 4)
Route has 13 retrosynthetic steps.



## 11. Save Results

We save two files to `tutorial_results/22_recursive_solver/`:
- **`result.json`**: SMILES, status, timing, cache statistics, and the full
  route graph (flat adjacency dict).
- **`route.svg`**: visual representation of the route using SynPlanner's
  built-in SVG renderer (same style as tutorial 05).

The SVG is only saved if a route was found.  The JSON is always saved so
failed attempts are also recorded.

In [14]:
output_dir = Path("tutorial_results/22_recursive_solver")
output_dir.mkdir(parents=True, exist_ok=True)

result = {
    "smiles": target_smiles,
    "canonical": canonical,
    "status": status,
    "n_atoms": len(mol),
    "wall_time_s": round(wall_time, 2),
    "cache_entries": len(fragment_cache),
    "route_graph": route_graph,
}

result_path = output_dir / "result.json"
with open(result_path, "w") as f:
    json.dump(result, f, indent=2, default=str)
print(f"Saved to {result_path}")

if route_graph:
    # Convert to nested JSON and render as SVG (same renderer as tutorial 05).
    route_json = {0: route_graph_to_json(route_graph, canonical, building_blocks)}
    svg_str = get_route_svg_from_json(route_json, 0)
    svg_path = output_dir / "route.svg"
    with open(svg_path, "w") as f:
        f.write(svg_str)
    print(f"Route SVG saved to {svg_path}")

Saved to tutorial_results/17_recursive_solver/result.json
Route SVG saved to tutorial_results/17_recursive_solver/route.svg
